# Day 39 · 店铺前台挂件

**配套讲义**: [`days/day-39.md`](../days/day-39.md) ｜ **本地可跑，不需要 GPU**

做一个 Theme App Extension（App Block）：店铺前台右下角出现客服入口，支持**上传图片**、调用你的 Agent API、样式跟随主题。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w7.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys
print("python:", sys.version.split()[0])
for m in ("numpy", "PIL", "yaml", "pandas"):
    try:
        mod = __import__(m)
        print(f"  {m:7s} {getattr(mod, '__version__', 'ok')}")
    except ImportError:
        print(f"  {m:7s} ❌ 缺 → pip install {m}")
print("\n→ 本机没 GPU 不影响今天：今天只用纯 Python / numpy")

## 1. 看挂件的 schema（店主能调什么）

In [ ]:
from pathlib import Path

p = Path("../src/shopify/extensions/chat-widget/blocks/chat_widget.liquid")
text = p.read_text()
i, j = text.find("{% schema %}"), text.find("{% endschema %}")
print(text[i:j + 15] if i >= 0 else "没找到 schema 段")

## 2. 本地模拟：把挂件的请求打一遍

不装主题也能验后端 —— 用 curl 模拟挂件的请求。

In [ ]:
import json, base64, urllib.request

# 造一张 1x1 的小图做 payload
png_1x1 = base64.b64decode(
    "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8z8BQDwAEhQGAhKmMIQAAAABJRU5ErkJggg==")
payload = {"shop": "demo.myshopify.com",
           "message": "这件有货吗？",
           "images": ["data:image/png;base64," + base64.b64encode(png_1x1).decode()]}

print("挂件会发的 payload:")
print(json.dumps({**payload, "images": ["<base64 1x1 png>"]}, ensure_ascii=False, indent=2))
print("\n→ 后端要能处理：图片是可选的、可能是 data URL、可能有多张")

## 3. 首屏性能自查（审核会看）

- [ ] 脚本 `async` 或 `defer`
- [ ] 挂件容器延后挂载（`DOMContentLoaded` 之后）
- [ ] CSS 用 inline scoped 样式，不引外部大文件
- [ ] 有 `design_mode` 早退分支

In [ ]:
perf_checklist = {
    "脚本 async/defer": None,
    "延后挂载": None,
    "无外部大 CSS": None,
    "design_mode 早退": None,
    "图片大小限制": None,
}
print("逐项打勾（True/False），W8 Day 45 审核自查会再用一次")

## 验收清单

- [ ] 在真实商品页上传一张商品图能收到回复（这是 M7 的关键一步）
- [ ] 挂件样式跟随主题（改 `accent_color` 生效）
- [ ] 主题编辑器里能配置，且**设计模式下不报错**
- [ ] 底部有 AI 免责声明

**卡住了？** 回看 [`days/day-39.md`](../days/day-39.md) 第五节「容易踩的坑」。

> **明天**：`days/day-40.md` —— Webhook 与索引增量同步